# Walmart M5 Feature Engineering

## Business Objective

Notebook 01 established the **structure and quality** of the source data. Notebook 02 explored **business and forecasting behavior**. This notebook converts those findings into a **model-ready feature table** for demand forecasting.

This notebook builds:

- demand lag features
- rolling demand statistics
- recent zero-sales behavior
- calendar and seasonality features
- Walmart event and SNAP features
- U.S. holiday features
- price features
- weather features
- economic features
- leakage checks
- final processed feature data for Notebook 04

> **Important:** Forecasting features must only use information available **before the target date**. Lag and rolling features are therefore shifted backward before calculation.

## 1. Import Libraries

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## 2. Define Project Paths

The notebook reads the Walmart M5 source files plus the external datasets created by the extraction scripts. The final feature data is written to `data/processed`.

In [2]:
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

CALENDAR_FILE = RAW_DIR / "calendar.csv"
PRICES_FILE = RAW_DIR / "sell_prices.csv"
SALES_FILE = RAW_DIR / "sales_train_evaluation.csv"

FRED_FILE = EXTERNAL_DIR / "fred_economic_data.csv"
HOLIDAYS_FILE = EXTERNAL_DIR / "us_holidays.csv"
WEATHER_FILE = EXTERNAL_DIR / "weather_history.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 3. Development Configuration

The full M5 evaluation file contains **30,490 product-store series** and **1,941 daily sales columns**. Melting everything at once would create roughly **59 million rows**, so this notebook works one store at a time during development.

Set `MAX_ITEMS = None` when you want every product in the selected store.

In [3]:
STORE_ID = "CA_1"
MAX_ITEMS = 200

LAGS = [1, 7, 14, 28]
ROLLING_WINDOWS = [7, 28]

## 4. Load Source Data

In [4]:
calendar = pd.read_csv(CALENDAR_FILE)
prices = pd.read_csv(PRICES_FILE)
sales = pd.read_csv(SALES_FILE)

fred = pd.read_csv(FRED_FILE)
holidays = pd.read_csv(HOLIDAYS_FILE)
weather = pd.read_csv(WEATHER_FILE)

calendar["date"] = pd.to_datetime(calendar["date"])
fred["date"] = pd.to_datetime(fred["date"])
holidays["date"] = pd.to_datetime(holidays["date"])
weather["time"] = pd.to_datetime(weather["time"])

for name, df in {
    "calendar": calendar,
    "prices": prices,
    "sales": sales,
    "fred": fred,
    "holidays": holidays,
    "weather": weather,
}.items():
    print(f"{name.upper():10} {df.shape}")

CALENDAR   (1969, 14)
PRICES     (6841121, 4)
SALES      (30490, 1947)
FRED       (198, 3)
HOLIDAYS   (65, 2)
WEATHER    (5907, 7)


## 5. Filter the Sales Source

We keep only the selected store and optionally limit the number of products while developing.

In [5]:
store_sales = sales.loc[
    sales["store_id"] == STORE_ID
].copy()

if MAX_ITEMS is not None:
    selected_items = (
        store_sales["item_id"]
        .drop_duplicates()
        .sort_values()
        .head(MAX_ITEMS)
    )

    store_sales = store_sales.loc[
        store_sales["item_id"].isin(selected_items)
    ].copy()

print("Selected store:", STORE_ID)
print("Product-store series:", len(store_sales))

Selected store: CA_1
Product-store series: 200


## 6. Reshape Sales from Wide to Long

The original `d_1 ... d_1941` columns are converted into one row per date-product-store observation.

In [6]:
id_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
]

day_columns = [
    column
    for column in store_sales.columns
    if column.startswith("d_")
]

sales_long = store_sales.melt(
    id_vars=id_columns,
    value_vars=day_columns,
    var_name="d",
    value_name="units_sold",
)

print("Long sales shape:", sales_long.shape)
sales_long.head()

Long sales shape: (388200, 8)


,id,item_id,dept_id,cat_id,store_id,state_id,d,units_sold
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3
1,FOODS_1_002_CA_1_evaluation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1,0
2,FOODS_1_003_CA_1_evaluation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,d_1,0
3,FOODS_1_004_CA_1_evaluation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,d_1,0
4,FOODS_1_005_CA_1_evaluation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1,3


## 7. Attach Calendar Information

In [7]:
calendar_columns = [
    "d",
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI",
]

feature_df = sales_long.merge(
    calendar[calendar_columns],
    on="d",
    how="left",
    validate="many_to_one",
)

feature_df = (
    feature_df
    .sort_values(["item_id", "store_id", "date"])
    .reset_index(drop=True)
)

feature_df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,units_sold,date,wm_yr_wk,...,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,2,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,3,1,2011,NaN,NaN,NaN,NaN,0,0,0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,4,2,2011,NaN,NaN,NaN,NaN,1,1,0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,5,2,2011,NaN,NaN,NaN,NaN,1,0,1


## 8. Validate the Base Feature Table

In [8]:
if feature_df["date"].isna().any():
    raise ValueError("Some sales rows did not match a calendar date.")

if feature_df["units_sold"].isna().any():
    raise ValueError("Missing sales values were found.")

if (feature_df["units_sold"] < 0).any():
    raise ValueError("Negative sales values were found.")

duplicate_keys = feature_df.duplicated(
    subset=["date", "item_id", "store_id"]
).sum()

if duplicate_keys > 0:
    raise ValueError(
        f"Found {duplicate_keys:,} duplicate date-item-store rows."
    )

print("PASS: base feature table")

PASS: base feature table


## 9. Historical Demand Lag Features

`lag_1`, `lag_7`, `lag_14`, and `lag_28` show earlier sales for the same item-store series.

In [9]:
series_key = ["item_id", "store_id"]

for lag in LAGS:
    feature_df[f"lag_{lag}"] = (
        feature_df
        .groupby(series_key)["units_sold"]
        .shift(lag)
    )

feature_df[
    [
        "date",
        "item_id",
        "store_id",
        "units_sold",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
    ]
].head(35)

,date,item_id,store_id,units_sold,lag_1,lag_7,lag_14,lag_28
0,2011-01-29,FOODS_1_001,CA_1,3,NaN,NaN,NaN,NaN
1,2011-01-30,FOODS_1_001,CA_1,0,3.0,NaN,NaN,NaN
2,2011-01-31,FOODS_1_001,CA_1,0,0.0,NaN,NaN,NaN
3,2011-02-01,FOODS_1_001,CA_1,1,0.0,NaN,NaN,NaN
4,2011-02-02,FOODS_1_001,CA_1,4,1.0,NaN,NaN,NaN
5,2011-02-03,FOODS_1_001,CA_1,2,4.0,NaN,NaN,NaN
6,2011-02-04,FOODS_1_001,CA_1,0,2.0,NaN,NaN,NaN
7,2011-02-05,FOODS_1_001,CA_1,2,0.0,3.0,NaN,NaN
8,2011-02-06,FOODS_1_001,CA_1,0,2.0,0.0,NaN,NaN
9,2011-02-07,FOODS_1_001,CA_1,0,0.0,0.0,NaN,NaN


## 10. Rolling Demand Features

The sales series is shifted by one day **before** rolling calculations so the current target is never used to predict itself.

In [10]:
shifted_sales = (
    feature_df
    .groupby(series_key)["units_sold"]
    .shift(1)
)

for window in ROLLING_WINDOWS:
    grouped_shifted = shifted_sales.groupby(
        [feature_df["item_id"], feature_df["store_id"]]
    )

    feature_df[f"rolling_mean_{window}"] = (
        grouped_shifted
        .rolling(window, min_periods=1)
        .mean()
        .reset_index(level=[0, 1], drop=True)
    )

    feature_df[f"rolling_std_{window}"] = (
        grouped_shifted
        .rolling(window, min_periods=2)
        .std()
        .reset_index(level=[0, 1], drop=True)
    )

feature_df[
    [
        "date",
        "item_id",
        "units_sold",
        "rolling_mean_7",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28",
    ]
].head(35)

,date,item_id,units_sold,rolling_mean_7,rolling_mean_28,rolling_std_7,rolling_std_28
0,2011-01-29,FOODS_1_001,3,NaN,NaN,NaN,NaN
1,2011-01-30,FOODS_1_001,0,3.000000,3.000000,NaN,NaN
2,2011-01-31,FOODS_1_001,0,1.500000,1.500000,2.121320,2.121320
3,2011-02-01,FOODS_1_001,1,1.000000,1.000000,1.732051,1.732051
4,2011-02-02,FOODS_1_001,4,1.000000,1.000000,1.414214,1.414214
5,2011-02-03,FOODS_1_001,2,1.600000,1.600000,1.816590,1.816590
6,2011-02-04,FOODS_1_001,0,1.666667,1.666667,1.632993,1.632993
7,2011-02-05,FOODS_1_001,2,1.428571,1.428571,1.618347,1.618347
8,2011-02-06,FOODS_1_001,0,1.285714,1.500000,1.496026,1.511858
9,2011-02-07,FOODS_1_001,0,1.285714,1.333333,1.496026,1.500000


## 11. Recent Zero-Sales Behavior

Notebook 02 showed substantial intermittent demand. This feature measures the fraction of the previous 28 days with zero sales.

In [11]:
zero_indicator = (
    feature_df["units_sold"]
    .eq(0)
    .astype(int)
)

shifted_zero = zero_indicator.groupby(
    [feature_df["item_id"], feature_df["store_id"]]
).shift(1)

feature_df["zero_sales_share_28"] = (
    shifted_zero
    .groupby([feature_df["item_id"], feature_df["store_id"]])
    .rolling(28, min_periods=1)
    .mean()
    .reset_index(level=[0, 1], drop=True)
)

## 12. Calendar and Seasonality Features

In [12]:
feature_df["day_of_week"] = feature_df["date"].dt.dayofweek
feature_df["day_of_month"] = feature_df["date"].dt.day
feature_df["week_of_year"] = (
    feature_df["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)
feature_df["month"] = feature_df["date"].dt.month
feature_df["year"] = feature_df["date"].dt.year
feature_df["is_weekend"] = (
    feature_df["day_of_week"] >= 5
).astype(int)

## 13. M5 Event Features

In [13]:
feature_df["has_event_1"] = (
    feature_df["event_name_1"].notna().astype(int)
)

feature_df["has_event_2"] = (
    feature_df["event_name_2"].notna().astype(int)
)

feature_df["has_any_event"] = (
    feature_df["has_event_1"] | feature_df["has_event_2"]
).astype(int)

## 14. SNAP Feature

The appropriate state-specific M5 SNAP column is converted into one general feature called `snap`.

In [14]:
state_id = feature_df["state_id"].iloc[0]

snap_column_map = {
    "CA": "snap_CA",
    "TX": "snap_TX",
    "WI": "snap_WI",
}

snap_column = snap_column_map[state_id]

feature_df["snap"] = feature_df[snap_column].astype(int)

print("Store:", STORE_ID)
print("State:", state_id)
print("SNAP source column:", snap_column)

Store: CA_1
State: CA
SNAP source column: snap_CA


## 15. External U.S. Holiday Features

In [15]:
holiday_lookup = holidays[
    ["date", "holiday_name"]
].copy()

feature_df = feature_df.merge(
    holiday_lookup,
    on="date",
    how="left",
    validate="many_to_one",
)

feature_df["is_us_holiday"] = (
    feature_df["holiday_name"]
    .notna()
    .astype(int)
)

## 16. Price Features

Price change features are calculated in the weekly price table first so `previous_sell_price` truly means the previous available Walmart week.

A separate `price_available` flag is created because missing price records are meaningful. We do **not** replace missing price with zero, because zero would incorrectly mean the product was free.

In [16]:
store_prices = prices.loc[
    prices["store_id"] == STORE_ID
].copy()

if MAX_ITEMS is not None:
    store_prices = store_prices.loc[
        store_prices["item_id"].isin(
            feature_df["item_id"].unique()
        )
    ].copy()

store_prices = store_prices.sort_values(
    ["store_id", "item_id", "wm_yr_wk"]
)

store_prices["previous_sell_price"] = (
    store_prices
    .groupby(["store_id", "item_id"])["sell_price"]
    .shift(1)
)

store_prices["price_change"] = (
    store_prices["sell_price"]
    - store_prices["previous_sell_price"]
)

store_prices["price_change_pct"] = (
    store_prices["price_change"]
    / store_prices["previous_sell_price"]
)

price_features = store_prices[
    [
        "store_id",
        "item_id",
        "wm_yr_wk",
        "sell_price",
        "previous_sell_price",
        "price_change",
        "price_change_pct",
    ]
].copy()

feature_df = feature_df.merge(
    price_features,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one",
)

feature_df["price_available"] = (
    feature_df["sell_price"]
    .notna()
    .astype(int)
)

## 17. Weather Features

Weather is joined by `date + state_id`.

In [17]:
weather_features = weather.rename(
    columns={
        "time": "date",
        "temperature_2m_max": "temperature_max",
        "temperature_2m_min": "temperature_min",
        "precipitation_sum": "precipitation",
        "snowfall_sum": "snowfall",
        "wind_speed_10m_max": "wind_speed_max",
    }
)[
    [
        "date",
        "state_id",
        "temperature_max",
        "temperature_min",
        "precipitation",
        "snowfall",
        "wind_speed_max",
    ]
].copy()

feature_df = feature_df.merge(
    weather_features,
    on=["date", "state_id"],
    how="left",
    validate="many_to_one",
)

## 18. Economic Features

FRED is monthly while sales are daily. The data is pivoted so each series becomes one column. A simple one-month availability lag is applied to reduce look-ahead risk.

In [18]:
fred_wide = (
    fred
    .pivot(
        index="date",
        columns="series_id",
        values="value",
    )
    .sort_index()
    .reset_index()
)

fred_wide["available_date"] = (
    fred_wide["date"]
    + pd.offsets.MonthBegin(1)
)

fred_available = (
    fred_wide
    .drop(columns="date")
    .sort_values("available_date")
)

feature_df = (
    pd.merge_asof(
        feature_df.sort_values("date"),
        fred_available,
        left_on="date",
        right_on="available_date",
        direction="backward",
    )
    .sort_values(["item_id", "store_id", "date"])
    .reset_index(drop=True)
)

## 19. Cyclical Calendar Features

Sine/cosine features preserve the circular nature of weekdays and months.

In [19]:
feature_df["dow_sin"] = np.sin(
    2 * np.pi * feature_df["day_of_week"] / 7
)
feature_df["dow_cos"] = np.cos(
    2 * np.pi * feature_df["day_of_week"] / 7
)

feature_df["month_sin"] = np.sin(
    2 * np.pi * feature_df["month"] / 12
)
feature_df["month_cos"] = np.cos(
    2 * np.pi * feature_df["month"] / 12
)

## 20. Select Final Feature Columns

In [20]:
final_columns = [
    "date",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "units_sold",

    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",

    "rolling_mean_7",
    "rolling_std_7",
    "rolling_mean_28",
    "rolling_std_28",
    "zero_sales_share_28",

    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "year",
    "is_weekend",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",

    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "has_any_event",

    "holiday_name",
    "is_us_holiday",
    "snap",

    "sell_price",
    "price_available",
    "previous_sell_price",
    "price_change",
    "price_change_pct",

    "temperature_max",
    "temperature_min",
    "precipitation",
    "snowfall",
    "wind_speed_max",

    "CPIAUCSL",
    "UNRATE",
    "FEDFUNDS",
]

model_features = feature_df[final_columns].copy()

print("Final feature shape:", model_features.shape)
model_features.head()

Final feature shape: (388200, 47)


,date,item_id,dept_id,cat_id,store_id,state_id,units_sold,lag_1,lag_7,lag_14,...,price_change,price_change_pct,temperature_max,temperature_min,precipitation,snowfall,wind_speed_max,CPIAUCSL,UNRATE,FEDFUNDS
0,2011-01-29,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,3,NaN,NaN,NaN,...,NaN,NaN,18.1,6.1,0.0,0.0,10.9,NaN,NaN,NaN
1,2011-01-30,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,0,3.0,NaN,NaN,...,NaN,NaN,13.1,7.6,1.0,0.0,16.8,NaN,NaN,NaN
2,2011-01-31,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,0,0.0,NaN,NaN,...,NaN,NaN,18.9,4.3,0.0,0.0,11.0,NaN,NaN,NaN
3,2011-02-01,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,1,0.0,NaN,NaN,...,NaN,NaN,17.2,4.9,0.0,0.0,10.9,221.187,9.1,0.17
4,2011-02-02,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,4,1.0,NaN,NaN,...,NaN,NaN,16.4,2.1,0.0,0.0,8.8,221.187,9.1,0.17


## 21. Missing-Value Review

Expected missing values can occur in early lags, the first previous-price record, pre-sale/inactive product periods, and early lagged economic observations. We inspect them rather than blindly using `dropna()`.

In [21]:
missing_summary = (
    model_features
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary[
    missing_summary > 0
]

event_name_2           387400
event_type_2           387400
holiday_name           377200
event_type_1           356600
event_name_1           356600
previous_sell_price     59976
price_change_pct        59976
price_change            59976
sell_price              58576
lag_28                   5600
lag_14                   2800
lag_7                    1400
FEDFUNDS                  600
UNRATE                    600
CPIAUCSL                  600
rolling_std_28            400
rolling_std_7             400
rolling_mean_7            200
lag_1                     200
zero_sales_share_28       200
rolling_mean_28           200
dtype: int64

### 21A. Investigate Missing Sell Prices

If any row has positive sales but no price, investigate the price join before modeling.

In [22]:
price_missing = model_features["sell_price"].isna()

print("Rows missing price:", f"{price_missing.sum():,}")
print("Percent missing:", f"{price_missing.mean():.2%}")

model_features.loc[
    price_missing,
    ["date", "item_id", "store_id", "units_sold"],
].head(20)

Rows missing price: 58,576
Percent missing: 15.09%


,date,item_id,store_id,units_sold
5823,2011-01-29,FOODS_1_004,CA_1,0
5824,2011-01-30,FOODS_1_004,CA_1,0
5825,2011-01-31,FOODS_1_004,CA_1,0
5826,2011-02-01,FOODS_1_004,CA_1,0
5827,2011-02-02,FOODS_1_004,CA_1,0
5828,2011-02-03,FOODS_1_004,CA_1,0
5829,2011-02-04,FOODS_1_004,CA_1,0
5830,2011-02-05,FOODS_1_004,CA_1,0
5831,2011-02-06,FOODS_1_004,CA_1,0
5832,2011-02-07,FOODS_1_004,CA_1,0


In [23]:
missing_price_sales = model_features.loc[
    model_features["sell_price"].isna(),
    "units_sold",
]

print("Missing-price rows:", len(missing_price_sales))
print(
    "Missing price + zero sales:",
    (missing_price_sales == 0).sum(),
)
print(
    "Missing price + positive sales:",
    (missing_price_sales > 0).sum(),
)

if (missing_price_sales > 0).any():
    raise ValueError(
        "Positive sales were found on rows with missing sell_price. "
        "Investigate the price join before modeling."
    )

print(
    "PASS: missing sell prices occur only on zero-sales rows."
)

Missing-price rows: 58576
Missing price + zero sales: 58576
Missing price + positive sales: 0
PASS: missing sell prices occur only on zero-sales rows.


## 22. Create the Modeling-Eligible Dataset

Rows without `lag_28` do not yet have the full intended demand history, so they are excluded from the modeling-eligible table.

In [24]:
model_ready = model_features.loc[
    model_features["lag_28"].notna()
].copy()

model_ready = (
    model_ready
    .sort_values(["date", "store_id", "item_id"])
    .reset_index(drop=True)
)

print(
    "Rows before lag-history filter:",
    f"{len(model_features):,}",
)
print(
    "Rows after lag-history filter:",
    f"{len(model_ready):,}",
)

Rows before lag-history filter: 388,200
Rows after lag-history filter: 382,600


## 23. Data-Leakage Sanity Check

Inspect one example series and verify that lag values correspond to prior dates rather than the current target.

In [25]:
example_item = model_ready["item_id"].iloc[0]

example = (
    model_features.loc[
        model_features["item_id"] == example_item,
        [
            "date",
            "units_sold",
            "lag_1",
            "lag_7",
            "rolling_mean_7",
        ],
    ]
    .sort_values("date")
    .head(40)
)

example

,date,units_sold,lag_1,lag_7,rolling_mean_7
0,2011-01-29,3,NaN,NaN,NaN
1,2011-01-30,0,3.0,NaN,3.000000
2,2011-01-31,0,0.0,NaN,1.500000
3,2011-02-01,1,0.0,NaN,1.000000
4,2011-02-02,4,1.0,NaN,1.000000
5,2011-02-03,2,4.0,NaN,1.600000
6,2011-02-04,0,2.0,NaN,1.666667
7,2011-02-05,2,0.0,3.0,1.428571
8,2011-02-06,0,2.0,0.0,1.285714
9,2011-02-07,0,0.0,0.0,1.285714


## 24. Final Feature Validation

In [26]:
final_duplicate_count = model_ready.duplicated(
    subset=["date", "item_id", "store_id"]
).sum()

if final_duplicate_count > 0:
    raise ValueError(
        f"Found {final_duplicate_count:,} duplicate feature keys."
    )

if (model_ready["units_sold"] < 0).any():
    raise ValueError("Negative target values found.")

if not model_ready["lag_28"].notna().all():
    raise ValueError(
        "Model-ready data still contains missing lag_28 values."
    )

print("PASS: unique date-item-store keys")
print("PASS: non-negative target")
print("PASS: full 28-day lag history")
print("PASS: feature table created")
print("Rows:", f"{len(model_ready):,}")
print("Columns:", model_ready.shape[1])

PASS: unique date-item-store keys
PASS: non-negative target
PASS: full 28-day lag history
PASS: feature table created
Rows: 382,600
Columns: 47


## 25. Save the Processed Feature Dataset

The output is saved separately by store so the same workflow can later be scaled across all ten stores.

In [27]:
item_suffix = (
    "all_items"
    if MAX_ITEMS is None
    else f"{MAX_ITEMS}_items"
)

OUTPUT_FILE = (
    PROCESSED_DIR
    / f"features_{STORE_ID}_{item_suffix}.csv"
)

model_ready.to_csv(
    OUTPUT_FILE,
    index=False,
)

print("Saved:", OUTPUT_FILE)
print("Shape:", model_ready.shape)

Saved: ..\data\processed\features_CA_1_200_items.csv
Shape: (382600, 47)


## 26. Feature Engineering Findings

### Historical Demand

- Lag features created: **1, 7, 14, and 28 days**
- Rolling features created: **7-day and 28-day rolling means and standard deviations**
- Intermittent-demand feature: **previous 28-day zero-sales share**
- Historical-demand features use only prior observations.

### Calendar / Events

- Calendar features: **day of week, day of month, week of year, month, year, weekend indicator**
- Cyclical features: **weekday and month sine/cosine encodings**
- M5 event features: **event metadata plus a general event indicator**
- Holiday feature: **U.S. holiday name and binary holiday indicator**
- SNAP feature: **state-specific SNAP flag converted to a general `snap` feature**

### Price

- Current weekly sell price
- Price-availability indicator
- Previous available weekly sell price
- Absolute price change
- Percentage price change
- Missing sell-price rows are retained when they occur during zero-sales periods; they are not converted to a false `$0` price.

### Weather

- Daily maximum temperature
- Daily minimum temperature
- Precipitation
- Snowfall
- Maximum wind speed

### Economics

- CPI (`CPIAUCSL`)
- Unemployment rate (`UNRATE`)
- Federal funds rate (`FEDFUNDS`)
- Economic observations are delayed by one month before joining to reduce look-ahead risk.

### Data Leakage Controls

- Lag features use only prior observations.
- Rolling statistics are calculated after shifting the sales series.
- Recent zero-sales behavior is calculated from prior days only.
- Economic observations are delayed before joining.
- Temporal train/validation/test splitting will be performed in Notebook 04 rather than random splitting.

### Development Scope

This notebook currently runs on **one selected store** and optionally a limited number of products for development. Once the logic is stable, the same feature-building logic should be moved into reusable pipeline code and scaled across all stores.

## 27. Next Step

The next notebook is `04_modeling.ipynb`.

Notebook 04 should:

1. load the feature-engineered dataset
2. define `units_sold` as the target
3. create **time-based** train/validation/test splits
4. establish a naive baseline
5. train forecasting models
6. compare forecasting metrics
7. inspect feature importance
8. evaluate the forecast horizon
9. select the strongest model for the later pricing system

> **Do not use a random train/test split for this time-series forecasting problem.**